# Task 2: Sentiment and Thematic Analysis

## Business Objective
Quantify review sentiment and identify recurring themes to uncover satisfaction
drivers and pain points for each Ethiopian bank.

## Tool Selection Rationale
**VADER** is chosen over TextBlob and DistilBERT because:
- Designed specifically for short texts like app reviews
- No GPU required — runs on any machine
- Fast processing for 1000+ reviews
- Produces compound confidence scores (-1 to +1)

**Author:** Sosina Ayele

In [2]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('Libraries imported successfully')

c:\KAIM\fintech-review-analytics\venv\Scripts\python.exe


## 1. Load Data

In [ ]:
paths = [
    '../data/bank_reviews.csv',
    r'c:\KAIM\fintech-review-analytics\data\bank_reviews.csv',
]

df = None
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path, encoding='utf-8-sig')
        print(f'Loaded from: {path}')
        break

df['review_id'] = range(1, len(df) + 1)
print(f'Shape: {df.shape}')
print(f'\nReviews per bank:')
print(df['bank'].value_counts().to_string())
df.head()

## 2. Text Preprocessing
Tokenization, stop-word removal, and lemmatization pipeline.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(preprocess_text)
print('Text preprocessing complete!')
print(f'Sample original: {df["review"].iloc[0][:100]}')
print(f'Sample cleaned:  {df["cleaned_review"].iloc[0][:100]}')

## 3. Sentiment Analysis Using VADER

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    if not isinstance(text, str):
        return 0.0
    return sia.polarity_scores(str(text))['compound']

def label_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    return 'neutral'

df['sentiment_score'] = df['review'].apply(get_sentiment_score)
df['sentiment_label'] = df['sentiment_score'].apply(label_sentiment)

print('=== Sentiment Distribution ===')
print(df['sentiment_label'].value_counts().to_string())
coverage = df['sentiment_label'].notna().sum() / len(df) * 100
print(f'\nCoverage: {coverage:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sentiment distribution
colors = {'positive': 'green', 'neutral': 'grey', 'negative': 'red'}
sentiment_counts = df['sentiment_label'].value_counts()
axes[0].bar(sentiment_counts.index,
            sentiment_counts.values,
            color=[colors[l] for l in sentiment_counts.index])
axes[0].set_title('Overall Sentiment Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

# Sentiment by bank
bank_sentiment = df.groupby('bank')['sentiment_score'].mean()
axes[1].bar(bank_sentiment.index, bank_sentiment.values,
            color=sns.color_palette('viridis', len(bank_sentiment)))
axes[1].set_title('Mean Sentiment Score by Bank', fontweight='bold')
axes[1].set_ylabel('Mean Sentiment Score')
axes[1].tick_params(axis='x', rotation=15)

# Sentiment by rating
rating_sentiment = df.groupby('rating')['sentiment_score'].mean()
axes[2].bar(rating_sentiment.index, rating_sentiment.values,
            color=sns.color_palette('coolwarm', len(rating_sentiment)))
axes[2].set_title('Mean Sentiment by Star Rating', fontweight='bold')
axes[2].set_xlabel('Star Rating')
axes[2].set_ylabel('Mean Sentiment Score')

plt.suptitle('Sentiment Analysis Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

## 4. Thematic Analysis

### Theme Grouping Logic:
Themes are identified by keyword matching across 5 business-relevant categories:
1. **Account Access Issues** — login, password, locked, OTP, authentication
2. **Transaction Performance** — transfer, payment, slow, failed, crash
3. **UI and Design** — interface, design, easy, navigation, update
4. **Customer Support** — support, service, help, response, complaint
5. **Feature Requests** — add, need, improve, missing, biometric

In [ ]:
THEME_KEYWORDS = {
    'Account Access Issues': [
        'login', 'password', 'locked', 'access', 'otp', 'pin',
        'authentication', 'sign', 'logout', 'session', 'expired'
    ],
    'Transaction Performance': [
        'transfer', 'transaction', 'slow', 'fast', 'payment',
        'send', 'receive', 'money', 'delay', 'failed', 'error',
        'loading', 'crash', 'freeze', 'stuck'
    ],
    'UI and Design': [
        'interface', 'design', 'ui', 'update', 'button', 'screen',
        'easy', 'simple', 'navigation', 'user', 'friendly', 'layout'
    ],
    'Customer Support': [
        'support', 'service', 'customer', 'help', 'response',
        'staff', 'agent', 'call', 'contact', 'complaint', 'solve'
    ],
    'Feature Requests': [
        'add', 'need', 'want', 'wish', 'please', 'request',
        'improve', 'better', 'missing', 'fingerprint', 'biometric'
    ],
}

def identify_theme(text):
    if not isinstance(text, str):
        return 'General'
    text_lower = text.lower()
    theme_scores = {
        theme: sum(1 for kw in keywords if kw in text_lower)
        for theme, keywords in THEME_KEYWORDS.items()
    }
    best_theme = max(theme_scores, key=theme_scores.get)
    return best_theme if theme_scores[best_theme] > 0 else 'General'

df['identified_theme'] = df['review'].apply(identify_theme)

print('=== Theme Distribution ===')
print(df['identified_theme'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Theme distribution overall
theme_counts = df['identified_theme'].value_counts()
axes[0].barh(theme_counts.index[::-1], theme_counts.values[::-1],
             color=sns.color_palette('viridis', len(theme_counts)))
axes[0].set_title('Overall Theme Distribution', fontweight='bold')
axes[0].set_xlabel('Count')

# Themes by bank
theme_bank = df.groupby(['bank', 'identified_theme']).size().unstack(fill_value=0)
theme_bank.plot(kind='bar', ax=axes[1], colormap='viridis')
axes[1].set_title('Themes by Bank', fontweight='bold')
axes[1].set_xlabel('Bank')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.suptitle('Thematic Analysis Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('thematic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

## 5. TF-IDF Keyword Extraction per Bank

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, bank in enumerate(df['bank'].unique()):
    bank_reviews = df[df['bank'] == bank]['cleaned_review'].dropna()
    bank_reviews = bank_reviews[bank_reviews.str.strip() != '']

    tfidf = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 2))
    matrix = tfidf.fit_transform(bank_reviews)
    scores = pd.DataFrame({
        'keyword': tfidf.get_feature_names_out(),
        'score': matrix.sum(axis=0).A1
    }).sort_values('score', ascending=False).head(10)

    axes[i].barh(scores['keyword'][::-1], scores['score'][::-1],
                 color=sns.color_palette('magma', len(scores)))
    axes[i].set_title(f'{bank}\nTop Keywords', fontweight='bold', fontsize=10)
    axes[i].set_xlabel('TF-IDF Score')

plt.suptitle('Top TF-IDF Keywords per Bank', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tfidf_per_bank.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

## 6. Save Results

In [ ]:
output = df[[
    'review_id', 'review', 'rating', 'date', 'bank', 'source',
    'sentiment_label', 'sentiment_score', 'identified_theme'
]].copy()

output.to_csv('../data/review_analysis.csv', index=False, encoding='utf-8-sig')
print(f'Results saved to data/review_analysis.csv')
print(f'Total reviews analyzed: {len(output)}')
coverage = output['sentiment_label'].notna().sum() / len(output) * 100
print(f'Sentiment coverage: {coverage:.1f}%')
print(output.head())

## 7. Key Findings

### Sentiment Findings:
- Sentiment scores assigned to 100% of reviews
- Higher star ratings correlate with more positive sentiment scores
- Banks vary in their mean sentiment — indicating different user satisfaction levels

### Thematic Findings:
- **Transaction Performance** is the most common theme across all banks
- **Account Access Issues** (login/password problems) is a major pain point
- **Feature Requests** indicate users want biometric login and better notifications
- **Customer Support** complaints are common in 1-star reviews
- **UI and Design** receives mixed feedback — positive for simplicity, negative for bugs

### Next Steps (Task 3):
- Store results in a database (PostgreSQL or SQLite)
- Build a dashboard to visualize insights
- Compare sentiment trends over time